In [2]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

print(torch.__version__)
print(torchvision.__version__)

2.11.0+cpu
0.26.0+cpu


In [3]:
# Check Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

# Define Hyper-parameters 
input_size = 28 * 28
num_classes = 10
num_epochs = 20
batch_size = 100
learning_rate = 0.01

Using device: cpu


In [4]:
transform = transforms.ToTensor()

full_train_dataset = torchvision.datasets.MNIST(
    root = "./data",
    train = True,
    transform = transform,
    download = True
)

test_dataset = torchvision.datasets.MNIST(
    root = "./data",
    train = False,
    transform = transform,
    download = True
)

train_size = 50000
val_size = 10000

train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

100.0%
100.0%
100.0%
100.0%

Train: 50000
Validation: 10000
Test: 10000


In [5]:
class Net1(nn.Module):
    # 20 -> 50 -> 20 -> output(10)
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 20),
            nn.ReLU(),
            nn.Linear(20, 50),
            nn.ReLU(),
            nn.Linear(50, 20),
            nn.ReLU(),
            nn.Linear(20, num_classes)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.model(x)


class Net2(nn.Module):
    # 10 -> 20 -> 30 -> 20 -> 10 -> output(10)
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 10),
            nn.ReLU(),
            nn.Linear(10, 20),
            nn.ReLU(),
            nn.Linear(20, 30),
            nn.ReLU(),
            nn.Linear(30, 20),
            nn.ReLU(),
            nn.Linear(20, 10),
            nn.ReLU(),
            nn.Linear(10, num_classes)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.model(x)


class Net3(nn.Module):
    # 10 -> 40 -> 70 -> 40 -> 10 -> output(10)
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 10),
            nn.ReLU(),
            nn.Linear(10, 40),
            nn.ReLU(),
            nn.Linear(40, 70),
            nn.ReLU(),
            nn.Linear(70, 40),
            nn.ReLU(),
            nn.Linear(40, 10),
            nn.ReLU(),
            nn.Linear(10, num_classes)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.model(x)

In [6]:
def train_model(model, train_loader, val_loader, num_epochs, learning_rate):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)

    train_losses = []
    val_losses = []

    start_time = time.time()

    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()

        avg_train_loss = running_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        model.eval()
        running_val_loss = 0.0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item()

        avg_val_loss = running_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")

    training_time = time.time() - start_time
    return model, train_losses, val_losses, training_time

In [7]:
def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [8]:
results = {}

model1 = Net1()
model1, train1, val1, time1 = train_model(model1, train_loader, val_loader, num_epochs, learning_rate)
results["Network 1"] = {
    "model": model1,
    "train_losses": train1,
    "val_losses": val1,
    "training_time": time1,
    "final_val_loss": val1[-1]
}

model2 = Net2()
model2, train2, val2, time2 = train_model(model2, train_loader, val_loader, num_epochs, learning_rate)
results["Network 2"] = {
    "model": model2,
    "train_losses": train2,
    "val_losses": val2,
    "training_time": time2,
    "final_val_loss": val2[-1]
}

model3 = Net3()
model3, train3, val3, time3 = train_model(model3, train_loader, val_loader, num_epochs, learning_rate)
results["Network 3"] = {
    "model": model3,
    "train_losses": train3,
    "val_losses": val3,
    "training_time": time3,
    "final_val_loss": val3[-1]
}

Epoch [1/20] - Train Loss: 2.2733 - Val Loss: 2.2147
Epoch [2/20] - Train Loss: 1.9793 - Val Loss: 1.5904
Epoch [3/20] - Train Loss: 1.1273 - Val Loss: 0.8370
Epoch [4/20] - Train Loss: 0.7058 - Val Loss: 0.6225
Epoch [5/20] - Train Loss: 0.5771 - Val Loss: 0.5411
Epoch [6/20] - Train Loss: 0.5073 - Val Loss: 0.4747
Epoch [7/20] - Train Loss: 0.4480 - Val Loss: 0.4205
Epoch [8/20] - Train Loss: 0.4030 - Val Loss: 0.3837
Epoch [9/20] - Train Loss: 0.3741 - Val Loss: 0.3593
Epoch [10/20] - Train Loss: 0.3532 - Val Loss: 0.3459
Epoch [11/20] - Train Loss: 0.3374 - Val Loss: 0.3283
Epoch [12/20] - Train Loss: 0.3231 - Val Loss: 0.3210
Epoch [13/20] - Train Loss: 0.3103 - Val Loss: 0.3045
Epoch [14/20] - Train Loss: 0.2987 - Val Loss: 0.2923
Epoch [15/20] - Train Loss: 0.2869 - Val Loss: 0.2829
Epoch [16/20] - Train Loss: 0.2760 - Val Loss: 0.2724
Epoch [17/20] - Train Loss: 0.2648 - Val Loss: 0.2625
Epoch [18/20] - Train Loss: 0.2557 - Val Loss: 0.2556
Epoch [19/20] - Train Loss: 0.2464 - 

In [9]:
for name, info in results.items():
    test_acc = evaluate_accuracy(info["model"], test_loader)
    print(f"{name}")
    print(f"  Final Validation Loss: {info['final_val_loss']:.4f}")
    print(f"  Training Time: {info['training_time']:.2f} seconds")
    print(f"  Test Accuracy: {test_acc:.2f}%")
    print()

Network 1
  Final Validation Loss: 0.2411
  Training Time: 188.76 seconds
  Test Accuracy: 92.96%

Network 2
  Final Validation Loss: 0.6205
  Training Time: 193.30 seconds
  Test Accuracy: 82.04%

Network 3
  Final Validation Loss: 0.4927
  Training Time: 190.05 seconds
  Test Accuracy: 86.35%



In [10]:
best_model_name = min(results, key=lambda x: results[x]["final_val_loss"])
best_val_loss = results[best_model_name]["final_val_loss"]

print("Best model based on validation error:")
print(best_model_name)
print(f"Validation Loss: {best_val_loss:.4f}")

Best model based on validation error:
Network 1
Validation Loss: 0.2411


## Answer

The model with the least validation error was **Network 1**, with a final validation loss of **0.2411**.

Training times:
- Network 1: **188.76 seconds**
- Network 2: **193.30 seconds**
- Network 3: **190.05 seconds**

In [1]:
import sys
print(sys.executable)

c:\Users\gerar\OneDrive\Documents\Universidad Enero 2026\CIIC5015-096\Project2CIIC5015\.venv\Scripts\python.exe
